In [9]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
import torch

PROMPT_MIDI_LOAD_PATH = "../data/1.mid"
CONTINUATION_MIDI_SAVE_PATH = "../data/1.mid"

model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)
torch.Tensor.cuda = lambda self, *args, **kwargs: self.to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

prompt = tokenizer.encode_from_file(
    PROMPT_MIDI_LOAD_PATH, return_tensors="pt"
).to(device)

continuation = model.generate(
    prompt.input_ids[..., :512],
    max_length=512,
    do_sample=True,
    temperature=0.97,
    top_p=0.95,
    use_cache=True,
)

midi_dict = tokenizer.decode(continuation[0].tolist())
midi_dict.to_midi().save(CONTINUATION_MIDI_SAVE_PATH)
